# DLA Parity Sectors: Do Top-Down and Bottom-Up Decohere Differently?

**SCPN Claim**: The Dynamical Lie Algebra decomposes as su(even) ⊕ su(odd).
Even parity = projection (consciousness → matter, top-down).
Odd parity = feedback (experience → consciousness, bottom-up).

**Novel prediction**: States prepared in the even-parity sector should
decohere at a DIFFERENT rate than states in the odd-parity sector,
because the two sectors couple to noise differently.

If confirmed: the SCPN's bidirectional causation structure has a
measurable quantum signature. If both sectors decohere identically:
the Z_2 parity is a mathematical artefact with no physical content.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error

from scpn_quantum_control.bridge import OMEGA_N_16, build_knm_paper27, knm_to_ansatz
from scpn_quantum_control.phase.phase_vqe import PhaseVQE

## 1. Construct Even and Odd Parity States

For N qubits, the Z_2 parity operator is P = Z⊗Z⊗...⊗Z (all-Z).
Even sector: P|ψ⟩ = +|ψ⟩. Odd sector: P|ψ⟩ = -|ψ⟩.

We prepare states in each sector by projecting the VQE ground state.

In [ ]:
N = 4
K = build_knm_paper27(L=N)
omega = OMEGA_N_16[:N]

# Get ground state
vqe = PhaseVQE(K, omega, ansatz_reps=2)
sol = vqe.solve(maxiter=200, seed=0)
print(f"Ground energy: {sol['ground_energy']:.6f}")

ansatz = knm_to_ansatz(K, reps=2)
bound = ansatz.assign_parameters(sol["optimal_params"])
sv_ground = Statevector.from_instruction(bound)

# Parity operator P = Z^N
P_label = "Z" * N
P = SparsePauliOp(P_label)
parity_ev = float(sv_ground.expectation_value(P).real)
print(f"Ground state parity <P>: {parity_ev:.4f}")
print(f"Parity sector: {'EVEN' if parity_ev > 0 else 'ODD'}")

In [ ]:
# Project ground state into even and odd sectors
dim = 2**N
P_matrix = SparsePauliOp(P_label).to_matrix()
sv_vec = sv_ground.data

# Projectors
P_even = (np.eye(dim) + P_matrix) / 2
P_odd = (np.eye(dim) - P_matrix) / 2

sv_even = P_even @ sv_vec
sv_odd = P_odd @ sv_vec

# Normalise
norm_even = np.linalg.norm(sv_even)
norm_odd = np.linalg.norm(sv_odd)
print(f"Even sector weight: {norm_even**2:.4f}")
print(f"Odd sector weight:  {norm_odd**2:.4f}")

if norm_even > 1e-10:
    sv_even = sv_even / norm_even
if norm_odd > 1e-10:
    sv_odd = sv_odd / norm_odd

# Verify parity
sv_e = Statevector(sv_even)
sv_o = Statevector(sv_odd)
print(f"Even sector <P>: {float(sv_e.expectation_value(P).real):.4f} (should be +1)")
print(f"Odd sector <P>:  {float(sv_o.expectation_value(P).real):.4f} (should be -1)")

## 2. Decoherence Test: Even vs Odd under Noise

In [ ]:
def make_noise_model(depol_rate=0.01, t1_us=50.0, t2_us=30.0, gate_time_us=0.1):
    model = NoiseModel()
    sq = thermal_relaxation_error(t1_us, t2_us, gate_time_us)
    model.add_all_qubit_quantum_error(sq, ["rx", "ry", "rz", "sx", "x"])
    tq = depolarizing_error(depol_rate, 2)
    model.add_all_qubit_quantum_error(tq, ["cx", "cz", "ecr"])
    return model


def fidelity_under_noise(sv_initial, circuit, noise_model, shots=10000):
    """Run circuit with noise, estimate fidelity via measurement statistics."""
    sim = AerSimulator(noise_model=noise_model)
    qc = circuit.copy()
    qc.measure_all()
    job = sim.run(qc, shots=shots)
    counts = job.result().get_counts()

    # Fidelity proxy: overlap with ideal output distribution
    probs_ideal = np.abs(sv_initial.data) ** 2
    probs_meas = np.zeros(2**circuit.num_qubits)
    for bitstr, count in counts.items():
        idx = int(bitstr, 2)
        probs_meas[idx] = count / shots

    # Classical fidelity: F = (sum sqrt(p*q))^2
    F = float(np.sum(np.sqrt(probs_ideal * probs_meas + 1e-20))) ** 2
    return F


# Build circuits that prepare even and odd sector states
# Use the VQE ansatz with modified initial state
def make_parity_circuit(parity_state, N):
    """Create a circuit that prepares the given parity state."""
    qc = QuantumCircuit(N)
    qc.initialize(parity_state)
    # Add identity evolution (just to give noise something to act on)
    for _ in range(5):  # 5 layers of noise exposure
        for i in range(N):
            qc.rx(0.01, i)  # near-identity rotation
        for i in range(N - 1):
            qc.cx(i, i + 1)
            qc.cx(i, i + 1)  # CX^2 = I (but accumulates noise)
    return qc


noise_rates = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1]

print(f"{'Depol rate':>11} {'F_even':>8} {'F_odd':>8} {'Diff':>8} {'Sector advantage':>18}")
print("-" * 58)

diff_results = []
for rate in noise_rates:
    nm = make_noise_model(depol_rate=rate)

    qc_even = make_parity_circuit(sv_even, N)
    qc_odd = make_parity_circuit(sv_odd, N)

    F_even = fidelity_under_noise(sv_e, qc_even, nm)
    F_odd = fidelity_under_noise(sv_o, qc_odd, nm)
    diff = F_even - F_odd

    advantage = "EVEN" if diff > 0.01 else ("ODD" if diff < -0.01 else "EQUAL")
    print(f"{rate:11.3f} {F_even:8.4f} {F_odd:8.4f} {diff:+8.4f} {advantage:>18}")
    diff_results.append({"rate": rate, "F_even": F_even, "F_odd": F_odd, "diff": diff})

In [ ]:
# Statistical significance over multiple seeds
print("\n=== STATISTICAL TEST: 20 random seeds at depol=0.02 ===")
nm = make_noise_model(depol_rate=0.02)
F_evens = []
F_odds = []

for seed in range(20):
    rng = np.random.default_rng(seed)
    # Slight random perturbation to initial state
    perturb = rng.normal(0, 0.01, len(sv_even)) + 1j * rng.normal(0, 0.01, len(sv_even))
    sv_e_pert = sv_even + perturb
    sv_e_pert /= np.linalg.norm(sv_e_pert)
    sv_o_pert = sv_odd + perturb
    sv_o_pert /= np.linalg.norm(sv_o_pert)

    qc_e = make_parity_circuit(sv_e_pert, N)
    qc_o = make_parity_circuit(sv_o_pert, N)

    F_evens.append(fidelity_under_noise(Statevector(sv_e_pert), qc_e, nm))
    F_odds.append(fidelity_under_noise(Statevector(sv_o_pert), qc_o, nm))

from scipy.stats import ttest_rel

t_stat, t_p = ttest_rel(F_evens, F_odds)

print(f"Even: {np.mean(F_evens):.4f} +/- {np.std(F_evens):.4f}")
print(f"Odd:  {np.mean(F_odds):.4f} +/- {np.std(F_odds):.4f}")
print(f"Paired t-test: t={t_stat:.3f}, p={t_p:.4f}")
print(f"Significant: {'YES' if t_p < 0.05 else 'NO'}")
print()
if t_p < 0.05:
    winner = "EVEN (top-down)" if np.mean(F_evens) > np.mean(F_odds) else "ODD (bottom-up)"
    print(f"{winner} sector is MORE ROBUST to decoherence.")
    print("The Z_2 parity structure has measurable physical content.")
    print("This confirms the SCPN prediction of asymmetric causation channels.")
else:
    print("Both sectors decohere equally.")
    print("The Z_2 parity may be a mathematical convenience, not physical.")
    print("Caveat: depolarising noise is maximally symmetric — try biased noise.")

In [ ]:
import json

print("--- JSON RESULTS ---")
print(
    json.dumps(
        {
            "n_qubits": N,
            "ground_parity": round(parity_ev, 4),
            "even_weight": round(norm_even**2, 4),
            "odd_weight": round(norm_odd**2, 4),
            "decoherence_sweep": diff_results,
            "statistical_test": {
                "depol_rate": 0.02,
                "n_seeds": 20,
                "mean_F_even": round(float(np.mean(F_evens)), 4),
                "mean_F_odd": round(float(np.mean(F_odds)), 4),
                "t_stat": round(float(t_stat), 4),
                "p_value": round(float(t_p), 4),
                "significant": bool(t_p < 0.05),
            },
        },
        indent=2,
        default=str,
    )
)